# TruthLens AI — LIAR Dataset Deep-Dive EDA
This notebook performs exploratory data analysis on the LIAR dataset (William Yang Wang, ACL 2017).

### Topics Covered:
- 6-class truth rating distribution across train, valid, and test splits
- Statement text length distributions
- Metadata analysis: Speaker distributions, party affiliation, contexts/venues
- Historical truth credit metrics
- Cross-split statement leakage
- Proposed 3-way label mappings for TruthLens AI

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath("../src"))
from liar_loader import load_all_liar_splits, LIAR_COLUMNS, LIAR_6_LABELS
from preprocessing import map_liar_label_to_3way

sns.set_theme(style="whitegrid")
liar_splits = load_all_liar_splits("../data/raw/liar")
df_train = liar_splits["train"]
df_valid = liar_splits["valid"]
df_test = liar_splits["test"]

print(f"LIAR loaded: Train={len(df_train):,}, Valid={len(df_valid):,}, Test={len(df_test):,}")

## 1. 6-Class Label Distribution Across Splits

In [ ]:
split_dfs = []
for name, df in liar_splits.items():
    s_counts = df["label"].value_counts(normalize=True).rename(name)
    split_dfs.append(s_counts)

df_label_comp = pd.concat(split_dfs, axis=1).reindex(LIAR_6_LABELS) * 100
display(df_label_comp.round(2))

df_label_comp.plot(kind="bar", figsize=(10, 5), colormap="Set2")
plt.title("LIAR Label Percentage Across Splits")
plt.ylabel("Percentage (%)")
plt.xlabel("Truth-O-Meter Rating")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Statement Length Statistics

In [ ]:
df_train["word_count"] = df_train["statement"].apply(lambda s: len(s.split()))
print(df_train["word_count"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(df_train["word_count"], bins=30, color="#d95f02", kde=True)
plt.title("LIAR Train Statement Word Count Distribution")
plt.xlim(0, 60)
plt.xlabel("Word Count")
plt.tight_layout()
plt.show()

## 3. Top Speakers and Party Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 speakers
top_speakers = df_train["speaker"].value_counts().head(10)
sns.barplot(x=top_speakers.values, y=top_speakers.index, ax=ax1, palette="Blues_r")
ax1.set_title("Top 10 Speakers in Train Split")
ax1.set_xlabel("Number of Statements")

# Top parties
top_parties = df_train["party_affiliation"].value_counts().head(5)
sns.barplot(x=top_parties.index, y=top_parties.values, ax=ax2, palette="Greens_r")
ax2.set_title("Statements by Political Party")
ax2.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Metadata Completeness & Missing Fields

In [ ]:
empty_counts = {}
for col in df_train.columns:
    empty_cnt = (df_train[col].astype(str).str.strip() == "").sum()
    empty_counts[col] = (empty_cnt / len(df_train)) * 100

pd.Series(empty_counts).sort_values(ascending=False).plot(kind="bar", figsize=(10, 4), color="#e41a1c")
plt.title("LIAR Metadata Sparsity (Missing / Empty % in Train)")
plt.ylabel("Percentage Missing (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Cross-Split Statement Leakage

In [ ]:
tr_stmts = set(df_train["statement"].str.strip().str.lower())
val_stmts = set(df_valid["statement"].str.strip().str.lower())
te_stmts = set(df_test["statement"].str.strip().str.lower())

print("Train & Valid overlap:", len(tr_stmts.intersection(val_stmts)))
print("Train & Test overlap:", len(tr_stmts.intersection(te_stmts)))
print("Valid & Test overlap:", len(val_stmts.intersection(te_stmts)))

overlap_sample = list(tr_stmts.intersection(val_stmts))
for s in overlap_sample:
    print(f"- Overlapping statement: {s}")

## 6. Mapping LIAR to 3-Way Schema

In [ ]:
df_train["mapped_label_3way"] = df_train["label"].apply(map_liar_label_to_3way)
print("Mapped 3-Way Distribution:")
print(df_train["mapped_label_3way"].value_counts())
print(df_train["mapped_label_3way"].value_counts(normalize=True) * 100)